In [1]:
# With much thanks to Islam S. for identifying that there was a missing import!
import os, re, math
from pathlib import Path
from datetime import datetime

import dotenv, yaml
dotenv.load_dotenv("configs/local.env")


def model_cache_path(model_id):
    home_dir = Path.home()
    cache_dir = os.environ.get('HF_HUB_CACHE',  home_dir / ".cache" / "huggingface" / "hub")

    model_hf = Path(cache_dir) / ("models--" + model_id.replace("/", "--"))
    model_ref = (model_hf / "refs" / "main").read_text(encoding="utf-8").strip()
    model_path = model_hf / "snapshots" / model_ref

    return model_path

In [2]:
DATASET_NAME = "ed-donner/pricer-data"
#BASE_MODEL = "meta-llama/Llama-3.1-8B"
BASE_MODEL = "meta-llama/Llama-3.2-1B"
# BASE_MODEL = "data/huggingface/hub/models--meta-llama--Llama-3.2-3B/snapshots/13afe5124825b4f3751f836b40dafda64c1ed062"
# BASE_MODEL = "data/huggingface/hub/models--meta-llama--Llama-3.2-1B/snapshots/4e20de362430cd3b72f300e6b0f18e50e7166e08"

PROJECT_NAME = "pricer"

#RUN_NAME = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
RUN_NAME = "2025-08-11"

# Run name for saving the model in the hub
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_ID = f"{os.environ['HF_USER']}/{PROJECT_RUN_NAME}"

In [3]:
# Optonal: offline mode

os.environ['HF_HUB_OFFLINE'] = 'True'
os.environ['WANDB_MODE'] = 'offline'
BASE_MODEL = model_cache_path(BASE_MODEL)

In [4]:
#from huggingface_hub import login
import wandb
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt

#login(os.environ['HF_TOKEN'], add_to_git_credential=True)

# Configure Weights & Biases to record against our project
wandb.init(project=PROJECT_NAME, name=RUN_NAME)
# wandb.login(key=os.environ['WANDB_API_KEY'], force=True)

os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "checkpoint" # if LOG_TO_WANDB else "end"
os.environ["WANDB_WATCH"] = "gradients"

In [5]:
from tqdm import tqdm
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig, EarlyStoppingCallback
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

⚙️  Running in WANDB offline mode


In [6]:
dataset = load_dataset(DATASET_NAME)
train_dataset, temp_dataset = dataset['train'], dataset['test']

eval_test_split = temp_dataset.train_test_split(test_size=0.5, seed=42)
eval_dataset = eval_test_split["train"]  # 验证集
test_dataset = eval_test_split["test"]   # 测试集

print(test_dataset[0])

Using the latest cached version of the dataset since ed-donner/pricer-data couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'default' at data/huggingface/datasets/ed-donner___pricer-data/default/0.0.0/f38df5f756eafe7e331348b26a823e4d706f4a08 (last modified on Tue Aug 12 23:37:21 2025).


{'text': 'How much does this cost to the nearest dollar?\n\nMatte Black Hoop Drop Step Side Nerf Bars Rail Running Boards For 07-18 Chevy Silverado/GMC Sierra Crew Cab\nApplication For Chevy Silverado 1500 / 2500 / 2500 HD / 3500 HD Crew Cab / GMC Sierra 1500 / 2500 HD / 3500 HD Crew Cab / GMC Sierra 1500 / 2500 HD / 3500 HD Denali Crew Cab / Chevy Silverado / GMC Sierra 2500 HD / 3500 HD Crew Cab Models With Diesel Turbocharged Engine ( Attention Note Rocker Panel Mount For Installation / Will Only Fit Models With 4 Full Size Door That Open In The Same Fashion / Will Not Fit 2011 & Up Diesel Models With DEF Tanks ) Drop Step Side Nerf\n\nPrice is $', 'price': 329.0}


In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,

    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
# quant_config = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:.1f} GB")

#print("named_modules:", list(base_model.named_modules()))
#for name, module in base_model.named_modules():
#    if any(x in name for x in ["proj", "fc"]):
#        print(name)

Memory footprint: 1.0 GB


In [9]:
# Train parameters

# Hyperparameters for QLoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# Hyperparameters for Training
EPOCHS = 2 # you can do more epochs if you wish, but only 1 is needed - more is probably overkill
BATCH_SIZE = 4 # on an A100 box this can go up to 16
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 1e-4
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"

# Admin config - note that SAVE_STEPS is how often it will upload to the hub
# I've changed this from 5000 to 2000 so that you get more frequent saves
STEPS = 50
SAVE_STEPS = 2000

In [10]:
# First, specify the configuration parameters for LoRA
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [11]:
# Next, specify the general configuration parameters for training

output_dir = Path("data") / PROJECT_RUN_NAME

train_parameters = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    #eval_strategy="no",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    logging_steps=STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=False,
    bf16=True, # False for cpu
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb", # if LOG_TO_WANDB else None; "tensorboard"
    run_name=RUN_NAME,
    dataset_text_field="text",
    save_strategy="steps",
    #hub_strategy="every_save",
    #push_to_hub=True,
    #hub_model_id=HUB_MODEL_ID,
    #hub_private_repo=True,

    completion_only_loss=True,

    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    restore_callback_states_from_checkpoint=True,
)

In [12]:
# And now, the Supervised Fine Tuning Trainer will carry out the fine-tuning
# Given these 2 sets of configuration parameters
# The latest version of trl is showing a warning about labels - please ignore this warning
# But let me know if you don't see good training results (loss coming down).

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_dataset,
    peft_config=lora_parameters,
    args=train_parameters,

    eval_dataset=eval_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [13]:
last_ckpt = get_last_checkpoint(output_dir)

print("==> fine_tuning start: {}, {}".format(datetime.now().astimezone().strftime("%FT%T%:z"), last_ckpt))

# Fine-tune!
fine_tuning.train(resume_from_checkpoint=last_ckpt if last_ckpt else None)

print("<== fine_tuning end:", datetime.now().astimezone().strftime("%FT%T%:z"))

==> fine_tuning start: 2025-08-13T10:14:00+08:00, data/pricer-2025-08-11/checkpoint-100000


wandb: WARNING URL not available in offline run


Step,Training Loss,Validation Loss
102000,2.178300,2.202695
104000,1.991200,2.198473


wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-102000)... Done. 0.3s
wandb: WARNING URL not available in offline run
wandb: Adding directory to artifact (./data/pricer-2025-08-11/checkpoint-104000)... Done. 0.5s
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


<== fine_tuning end: 2025-08-13T10:36:35+08:00


In [15]:
fine_tuning.save_model(output_dir)

# Push our fine-tuned model to Hugging Face
#fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
#print(f"Saved to the hub: {PROJECT_RUN_NAME}")

wandb.finish()

eval/loss,█▁
eval/mean_token_accuracy,█▁
eval/num_tokens,▁█
eval/runtime,▁█
eval/samples_per_second,█▁
eval/steps_per_second,█▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇█████
train/global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/grad_norm,▃▄▁▂▂▂▃▃▅▅▁▃▃▂▃▄▁▂▄█▄▂▄▁▃▃▂▃▇▂▂▂▂▃▂▂▄▂▆▃
train/learning_rate,███▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
train/loss,▁▃▇▅▆▆▅▅▅▄▅▃▅▅▇▇▅▆▆▃▄▅▅▇▄▃▄▃▄▇▅▇█▅▄▂▃▅▅▂
